# Genome Preprocessing and Dataset Construction

This notebook prepares the validated *Staphylococcus aureus* genome sequences
for sequence-based ciprofloxacin resistance prediction.

## Workflow

1. Load the genome retrieval manifest.
2. Perform final genome-level data quality checks.
3. Create genome-level train, validation, and test splits.
4. Prevent isolate-level leakage between splits.
5. Optionally evaluate genome relatedness using Mash/MinHash.
6. Generate fixed-length sequence windows independently within each split.
7. Track genomic coordinates and sequence quality for every window.
8. Convert sequence windows into one-hot encoded representations.
9. Prepare the resulting data for CNN training.

## Important Data-Splitting Principle

Genome-level splitting is performed **before** window generation.

Windows from the same genome must never appear across different dataset splits.
This prevents information leakage between training, validation, and test sets.

## Step 1 — Load Final Genome Manifest

The genome retrieval manifest links each phenotype-labeled genome to its
BV-BRC availability and local FASTA sequence.

In [3]:
import pandas as pd
from pathlib import Path

# Load final genome retrieval manifest

manifest_path = "../data/processed/genome_retrieval_manifest.csv"

manifest_df = pd.read_csv(
    manifest_path,
    dtype={"Genome ID": str}
)

print("Manifest shape:", manifest_df.shape)

print("\nColumns:")
print(manifest_df.columns.tolist())

print("\nFASTA availability:")
print(
    manifest_df["FASTA Available"].value_counts()
)

Manifest shape: (1205, 5)

Columns:
['Genome ID', 'Phenotype', 'Label', 'Metadata Available', 'FASTA Available']

FASTA availability:
FASTA Available
True     1138
False      67
Name: count, dtype: int64


## Step 2 — Define the Modeling Cohort

The modeling cohort contains only genomes with successfully retrieved FASTA
sequences.

The 67 genomes unavailable from BV-BRC remain documented in the retrieval
manifest but are excluded from sequence-based modeling.

In [4]:
# Select genomes with successfully retrieved FASTA files

model_df = manifest_df[
    manifest_df["FASTA Available"] == True
].copy()

print("Modeling genomes:", len(model_df))

print("\nLabel distribution:")
print(model_df["Label"].value_counts())

print("\nPhenotype distribution:")
print(model_df["Phenotype"].value_counts())

print("\nUnique Genome IDs:", model_df["Genome ID"].nunique())

Modeling genomes: 1138

Label distribution:
Label
0    664
1    474
Name: count, dtype: int64

Phenotype distribution:
Phenotype
('Susceptible',)    664
('Resistant',)      474
Name: count, dtype: int64

Unique Genome IDs: 1138


## Step 3 — Clean Phenotype Labels

The phenotype column is normalized to plain string labels so that the dataset
contains the intended categorical values:

- Susceptible
- Resistant

The binary `Label` column remains unchanged:
- 0 = Susceptible
- 1 = Resistant

In [5]:
# Normalize phenotype labels

model_df["Phenotype"] = (
    model_df["Phenotype"]
    .astype(str)
    .str.strip("()', ")
)

print("Phenotype distribution:")
print(model_df["Phenotype"].value_counts())

print("\nUnique phenotype values:")
print(model_df["Phenotype"].unique())

Phenotype distribution:
Phenotype
Susceptible    664
Resistant      474
Name: count, dtype: int64

Unique phenotype values:
<StringArray>
['Susceptible', 'Resistant']
Length: 2, dtype: str


## Step 4 — Genome-Level Train/Validation/Test Split

The modeling cohort is divided at the genome level using a stratified split:

- 70% training
- 15% validation
- 15% testing

Stratification preserves the Resistant/Susceptible proportion across splits.

The split is performed before genome window generation to prevent windows
originating from the same genome from appearing across different datasets.

In [6]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["Label"],
    random_state=42
)

# Second split: divide the temporary 30% equally
# → 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Label"],
    random_state=42
)

print("Train genomes:", len(train_df))
print("Validation genomes:", len(val_df))
print("Test genomes:", len(test_df))

print("\nTrain label distribution:")
print(train_df["Label"].value_counts())

print("\nValidation label distribution:")
print(val_df["Label"].value_counts())

print("\nTest label distribution:")
print(test_df["Label"].value_counts())

Train genomes: 796
Validation genomes: 171
Test genomes: 171

Train label distribution:
Label
0    464
1    332
Name: count, dtype: int64

Validation label distribution:
Label
0    100
1     71
Name: count, dtype: int64

Test label distribution:
Label
0    100
1     71
Name: count, dtype: int64


## Step 5 — Verify Genome-Level Split Integrity

The train, validation, and test sets must contain completely disjoint Genome IDs.

This prevents genomic windows originating from the same isolate from appearing
in multiple dataset partitions and causing data leakage.

In [7]:
# Verify that Genome IDs are completely disjoint

train_ids = set(train_df["Genome ID"])
val_ids = set(val_df["Genome ID"])
test_ids = set(test_df["Genome ID"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(val_test_overlap))

print("\nTotal unique Genome IDs across all splits:",
      len(train_ids | val_ids | test_ids))

print("Expected unique Genome IDs:", len(model_df))

if not train_val_overlap and not train_test_overlap and not val_test_overlap:
    print("\n✓ No genome-level leakage detected.")
else:
    print("\n✗ Genome overlap detected.")

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Total unique Genome IDs across all splits: 1138
Expected unique Genome IDs: 1138

✓ No genome-level leakage detected.


## Step 6 — Define Genome Windowing

Each genome is processed contig-by-contig rather than concatenated into a
single artificial sequence.

Each contig is divided into overlapping windows using:

- Window size: 500 bp
- Stride: 250 bp

For every generated window, we retain:

- Genome ID
- Contig ID
- Start coordinate
- End coordinate
- Strand
- Sequence
- Fraction of ambiguous bases (N)

Windows are generated only after the genome-level train/validation/test split,
ensuring that windows from the same isolate cannot cross dataset boundaries.

In [8]:
# Windowing configuration

WINDOW_SIZE = 500
STRIDE = 250

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)

Window size: 500
Stride: 250


## Step 7 — Parse Genome FASTA into Contigs

Genome FASTA files may contain multiple contigs. Each contig is treated as an
independent nucleotide sequence.

We parse the FASTA file into a dictionary mapping each contig identifier to
its nucleotide sequence.

Contigs are never concatenated across boundaries.

In [9]:
from pathlib import Path

def parse_fasta(fasta_path):
    """
    Parse a FASTA file into a dictionary:
    {contig_id: nucleotide_sequence}
    """
    contigs = {}
    current_id = None
    sequence_parts = []

    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if current_id is not None:
                    contigs[current_id] = "".join(sequence_parts)

                current_id = line[1:].split()[0]
                sequence_parts = []

            else:
                sequence_parts.append(line.upper())

        if current_id is not None:
            contigs[current_id] = "".join(sequence_parts)

    return contigs

In [10]:
from pathlib import Path

genome_dir = Path("../data/genomes")

print("Genome directory:", genome_dir)
print("Directory exists:", genome_dir.exists())

Genome directory: ..\data\genomes
Directory exists: True


In [11]:
# Test FASTA parsing on the representative genome

test_genome_id = "1280.15865"
test_fasta_path = genome_dir / f"{test_genome_id}.fna"

test_contigs = parse_fasta(test_fasta_path)

print("Genome ID:", test_genome_id)
print("Number of contigs:", len(test_contigs))

print("\nFirst 5 contigs:")
for contig_id, sequence in list(test_contigs.items())[:5]:
    print(contig_id, "—", len(sequence), "bp")

Genome ID: 1280.15865
Number of contigs: 26

First 5 contigs:
accn|LKNK01000018 — 87177 bp
accn|LKNK01000003 — 198837 bp
accn|LKNK01000016 — 83819 bp
accn|LKNK01000010 — 130579 bp
accn|LKNK01000006 — 24070 bp


## Step 8 — Define Genome Windowing Function

Each genome is processed contig-by-contig.

For each contig, overlapping windows are generated using:
- Window size: 500 bp
- Stride: 250 bp

Windows are never generated across contig boundaries.

Each window retains:
- Genome ID
- Contig ID
- Start coordinate
- End coordinate
- Strand
- Sequence
- Fraction of ambiguous bases (N)

Window generation occurs only after genome-level dataset splitting,
preventing windows from the same isolate from appearing across different
dataset partitions.

In [12]:
# Windowing configuration

WINDOW_SIZE = 500
STRIDE = 250

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)

Window size: 500
Stride: 250


In [13]:
def generate_windows(sequence, window_size=500, stride=250):
    """
    Generate overlapping windows from a single contig sequence.

    Windows are generated only within the contig.
    The final incomplete window is discarded.
    """
    windows = []

    for start in range(0, len(sequence) - window_size + 1, stride):
        end = start + window_size
        window_sequence = sequence[start:end]

        n_fraction = window_sequence.upper().count("N") / window_size

        windows.append({
            "Start": start,
            "End": end,
            "Sequence": window_sequence,
            "N_fraction": n_fraction
        })

    return windows

In [14]:
# Test window generation on one contig

first_contig_id, first_contig_sequence = next(iter(test_contigs.items()))

test_windows = generate_windows(
    first_contig_sequence,
    WINDOW_SIZE,
    STRIDE
)

print("Contig ID:", first_contig_id)
print("Contig length:", len(first_contig_sequence), "bp")
print("Number of windows:", len(test_windows))

print("\nFirst window:")
print(test_windows[0])

Contig ID: accn|LKNK01000018
Contig length: 87177 bp
Number of windows: 347

First window:
{'Start': 0, 'End': 500, 'Sequence': 'TCTTACTGCTGTTTTTTTAGGGATTTATGTCCCAGCCTCTTTTTGGTTTTAATTTTACAATAATTGCTTAGTATCTTTTAAATCAAATGTGCAATTGTTAATATAGCTCTATATTAGAGTGTTTTGGATGAAACATTTGCTATGATAAGGTAAAATTAAATTAAGAAAGTGCTTTAATGCTTTAACTGTATAAATCATTTAAATAGTTTTGCTACAACGAATATGTTATTATAAGTTACCAAAATATTTTAGAGAAAGAAATCAAACAAATAAATTGTATTTTGTTTGATGCTTTAATACAATGTGGATACTATTCATCGACTGTAAGGAGTGAGATGATATGAGAATAGAACGAGTAGATGATACAACTGTAAAATTGTTTATAACATATAGCGATATCGAGGCCCGTGGATTTAGTCGTGAAGATTTATGGACAAATCGCAAACGTGGCGAAGAATTCTTTTGGTCAATGATGGATGAAATTAACGAAGAAGAAGATT', 'N_fraction': 0.0}


In [15]:
# Inspect the first few window coordinates

for window in test_windows[:5]:
    print(
        "Start:", window["Start"],
        "| End:", window["End"],
        "| Length:", len(window["Sequence"]),
        "| N fraction:", window["N_fraction"]
    )

Start: 0 | End: 500 | Length: 500 | N fraction: 0.0
Start: 250 | End: 750 | Length: 500 | N fraction: 0.0
Start: 500 | End: 1000 | Length: 500 | N fraction: 0.0
Start: 750 | End: 1250 | Length: 500 | N fraction: 0.0
Start: 1000 | End: 1500 | Length: 500 | N fraction: 0.0


In [16]:
# Test windowing on several contigs

test_contigs = list(test_contigs.items())[:5]

for contig_id, sequence in test_contigs:
    windows = generate_windows(
        sequence,
        window_size=WINDOW_SIZE,
        stride=STRIDE
    )

    print(
        contig_id,
        "| Contig length:", len(sequence),
        "| Windows:", len(windows)
    )

accn|LKNK01000018 | Contig length: 87177 | Windows: 347
accn|LKNK01000003 | Contig length: 198837 | Windows: 794
accn|LKNK01000016 | Contig length: 83819 | Windows: 334
accn|LKNK01000010 | Contig length: 130579 | Windows: 521
accn|LKNK01000006 | Contig length: 24070 | Windows: 95


## Step 12 — Estimate Total Genome Windows

Before generating the complete window-level dataset, we first estimate the
number of windows that will be produced from the modeling cohort.

This provides an approximate dataset size and helps determine whether the
window-level data can be safely constructed in memory or should be processed
in batches.

For each genome:

1. Load the corresponding FASTA file.
2. Parse the genome into individual contigs.
3. Generate windows independently within each contig.
4. Count the resulting windows.
5. Sum the counts across all genomes in the modeling cohort.

The current windowing configuration is:

- Window size: **500 bp**
- Stride: **250 bp**
- Contigs are processed independently.
- Windows are generated only from genomes belonging to the modeling cohort.

No window-level data is saved at this stage; this step only estimates the
total number of windows that will be generated.

In [17]:
# Estimate total number of windows in the modeling cohort

total_windows = 0
genomes_with_files = 0

for genome_id in model_df["Genome ID"]:
    fasta_path = genome_dir / f"{genome_id}.fna"

    if not fasta_path.exists():
        continue

    contigs = parse_fasta(fasta_path)

    for sequence in contigs.values():
        total_windows += len(
            generate_windows(
                sequence,
                window_size=WINDOW_SIZE,
                stride=STRIDE
            )
        )

    genomes_with_files += 1

print("Genomes available:", genomes_with_files)
print("Estimated total windows:", total_windows)

Genomes available: 1138
Estimated total windows: 12701599


## Step 13 — Estimate Windows Per Dataset Split

The modeling cohort has already been divided at the genome level into
training, validation, and test sets.

Before constructing the complete window-level dataset, we estimate the
number of windows produced by each split.

This step is important because the number of windows can be very large,
especially when using overlapping 500 bp windows with a 250 bp stride.

The estimate will guide the subsequent dataset-construction strategy and
help avoid loading the complete window-level dataset into memory at once.

Windows remain associated with their source genome and are generated only
after the genome-level split, preserving the separation between training,
validation, and test genomes.

In [18]:
# Estimate the number of windows in each dataset split

def estimate_split_windows(split_df):
    total_windows = 0
    genomes_processed = 0

    for genome_id in split_df["Genome ID"]:
        fasta_path = genome_dir / f"{genome_id}.fna"

        if not fasta_path.exists():
            continue

        contigs = parse_fasta(fasta_path)

        for sequence in contigs.values():
            total_windows += len(
                generate_windows(
                    sequence,
                    window_size=WINDOW_SIZE,
                    stride=STRIDE
                )
            )

        genomes_processed += 1

    return genomes_processed, total_windows


train_genomes, train_windows = estimate_split_windows(train_df)
val_genomes, val_windows = estimate_split_windows(val_df)
test_genomes, test_windows = estimate_split_windows(test_df)

print("Training genomes:", train_genomes)
print("Training windows:", train_windows)

print("\nValidation genomes:", val_genomes)
print("Validation windows:", val_windows)

print("\nTest genomes:", test_genomes)
print("Test windows:", test_windows)

print("\nTotal windows:", train_windows + val_windows + test_windows)

Training genomes: 796
Training windows: 8878455

Validation genomes: 171
Validation windows: 1912079

Test genomes: 171
Test windows: 1911065

Total windows: 12701599


## Step 14 — Define Genome-Level Window Processing

Each genome is treated as one multiple-instance learning (MIL) bag.

The genome is first parsed into its individual contigs, and windows are
generated independently within each contig using the configured window size
and stride.

Each generated window retains its genomic provenance:

- Genome ID
- Phenotype
- Binary label
- Contig ID
- Start coordinate
- End coordinate
- Strand
- N fraction
- Sequence

The windows are therefore treated as instances belonging to a particular
genome rather than as independent training examples.

This distinction is essential because the final prediction target is the
genome-level antimicrobial-resistance phenotype.

In [19]:
def process_genome(genome_id):
    """
    Process one genome into its window-level MIL instances.

    Each genome remains a single MIL bag, while individual windows
    retain their genomic coordinates and source information.
    """

    # Locate FASTA file
    fasta_path = genome_dir / f"{genome_id}.fna"

    if not fasta_path.exists():
        raise FileNotFoundError(
            f"FASTA file not found for Genome ID: {genome_id}"
        )

    # Get genome-level label information
    genome_row = model_df[
        model_df["Genome ID"] == genome_id
    ]

    if len(genome_row) != 1:
        raise ValueError(
            f"Expected exactly one phenotype record for {genome_id}, "
            f"found {len(genome_row)}"
        )

    phenotype = genome_row.iloc[0]["Phenotype"]
    label = int(genome_row.iloc[0]["Label"])

    # Parse genome into contigs
    contigs = parse_fasta(fasta_path)

    windows = []

    # Generate windows independently for each contig
    for contig_id, sequence in contigs.items():

        contig_windows = generate_windows(
            sequence,
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        for window in contig_windows:

            windows.append({
                "Genome ID": genome_id,
                "Phenotype": phenotype,
                "Label": label,
                "Contig ID": contig_id,
                "Start": window["Start"],
                "End": window["End"],
                "Strand": "+",
                "Sequence": window["Sequence"],
                "N_fraction": window["N_fraction"]
            })

    return windows

In [20]:
test_genome_id = "1280.15865"

test_genome_windows = process_genome(test_genome_id)

print("Genome ID:", test_genome_id)
print("Total windows:", len(test_genome_windows))

print("\nFirst window:")
print(test_genome_windows[0])

print("\nLast window:")
print(test_genome_windows[-1])

Genome ID: 1280.15865
Total windows: 10885

First window:
{'Genome ID': '1280.15865', 'Phenotype': 'Susceptible', 'Label': 0, 'Contig ID': 'accn|LKNK01000018', 'Start': 0, 'End': 500, 'Strand': '+', 'Sequence': 'TCTTACTGCTGTTTTTTTAGGGATTTATGTCCCAGCCTCTTTTTGGTTTTAATTTTACAATAATTGCTTAGTATCTTTTAAATCAAATGTGCAATTGTTAATATAGCTCTATATTAGAGTGTTTTGGATGAAACATTTGCTATGATAAGGTAAAATTAAATTAAGAAAGTGCTTTAATGCTTTAACTGTATAAATCATTTAAATAGTTTTGCTACAACGAATATGTTATTATAAGTTACCAAAATATTTTAGAGAAAGAAATCAAACAAATAAATTGTATTTTGTTTGATGCTTTAATACAATGTGGATACTATTCATCGACTGTAAGGAGTGAGATGATATGAGAATAGAACGAGTAGATGATACAACTGTAAAATTGTTTATAACATATAGCGATATCGAGGCCCGTGGATTTAGTCGTGAAGATTTATGGACAAATCGCAAACGTGGCGAAGAATTCTTTTGGTCAATGATGGATGAAATTAACGAAGAAGAAGATT', 'N_fraction': 0.0}

Last window:
{'Genome ID': '1280.15865', 'Phenotype': 'Susceptible', 'Label': 0, 'Contig ID': 'accn|LKNK01000023', 'Start': 17750, 'End': 18250, 'Strand': '+', 'Sequence': 'GTCATGATTGGTAGAGGCATTTTCCATAATCCATTCGCTTTTGAAAAAGAACCACGCGAACACACAAGCAAAGAACTATTAGATTTGTTGAG

## Step 16 — Validate Window Integrity

The generated windows are validated before being used for model training.

The validation checks:

- Every window has the expected length of 500 bp.
- Window coordinates correspond to the sequence length.
- Only valid nucleotide characters (`A`, `C`, `G`, `T`, `N`) are present.
- The reported `N_fraction` agrees with the number of ambiguous bases.
- Windows do not extend beyond their source contig boundaries.

These checks ensure that the window-generation process produces valid,
well-defined genomic instances before numerical sequence representation and
model training.

In [21]:
# Validate window integrity for the test genome

valid_bases = set("ACGTN")

invalid_length = []
invalid_coordinates = []
invalid_bases = []
invalid_n_fraction = []

for window in test_genome_windows:

    sequence = window["Sequence"]
    start = window["Start"]
    end = window["End"]

    # Check sequence length
    if len(sequence) != WINDOW_SIZE:
        invalid_length.append(window)

    # Check coordinate length
    if end - start != WINDOW_SIZE:
        invalid_coordinates.append(window)

    # Check nucleotide characters
    if not set(sequence).issubset(valid_bases):
        invalid_bases.append(window)

    # Check N fraction
    expected_n_fraction = sequence.count("N") / len(sequence)

    if abs(window["N_fraction"] - expected_n_fraction) > 1e-9:
        invalid_n_fraction.append(window)


print("Total windows:", len(test_genome_windows))

print("Invalid sequence lengths:", len(invalid_length))
print("Invalid coordinates:", len(invalid_coordinates))
print("Invalid nucleotide characters:", len(invalid_bases))
print("Incorrect N fractions:", len(invalid_n_fraction))

if not any([
    invalid_length,
    invalid_coordinates,
    invalid_bases,
    invalid_n_fraction
]):
    print("\n✓ Window integrity validation passed.")
else:
    print("\n✗ Window integrity problems detected.")

Total windows: 10885
Invalid sequence lengths: 0
Invalid coordinates: 0
Invalid nucleotide characters: 0
Incorrect N fractions: 0

✓ Window integrity validation passed.


## Step 17 — Validate Window Coverage

The windowing procedure is further validated by comparing the genomic
sequence covered by the generated windows with the source contig lengths.

Because the windows use a 500 bp size and 250 bp stride, overlapping windows
are expected. Therefore, this check verifies the number and coordinate range
of windows rather than requiring every base to occur only once.

This confirms that windows remain within their respective contigs and that
the window-generation procedure is consistent across contigs.

In [22]:
# Validate window coverage for the test genome

coverage_errors = []

for contig_id, sequence in test_contigs:

    contig_length = len(sequence)

    contig_windows = [
        window for window in test_genome_windows
        if window["Contig ID"] == contig_id
    ]

    if not contig_windows:
        coverage_errors.append(
            (contig_id, "No windows generated")
        )
        continue

    # First window should start at 0
    if contig_windows[0]["Start"] != 0:
        coverage_errors.append(
            (contig_id, "First window does not start at 0")
        )

    # Every window must stay within the contig
    for window in contig_windows:
        if window["End"] > contig_length:
            coverage_errors.append(
                (contig_id, "Window extends beyond contig")
            )

    # Last window must not exceed the contig
    if contig_windows[-1]["End"] > contig_length:
        coverage_errors.append(
            (contig_id, "Last window exceeds contig length")
        )


print("Total contigs:", len(test_contigs))
print("Contigs with coverage errors:", len(coverage_errors))

if not coverage_errors:
    print("\n✓ Window coverage validation passed.")
else:
    print("\n✗ Coverage problems detected.")
    print(coverage_errors[:10])

Total contigs: 5
Contigs with coverage errors: 0

✓ Window coverage validation passed.


## Step 18 — Define One-Hot Sequence Encoding

Each 500 bp DNA window is converted into a numerical representation
suitable for the CNN.

Each nucleotide is represented using a four-dimensional one-hot vector:

- A → [1, 0, 0, 0]
- C → [0, 1, 0, 0]
- G → [0, 0, 1, 0]
- T → [0, 0, 0, 1]

Ambiguous nucleotide `N` is represented as:

- N → [0, 0, 0, 0]

The resulting representation for a 500 bp window has shape:

    (500, 4)

This representation preserves the nucleotide sequence while allowing the
CNN to learn local sequence patterns.

In [23]:
# Define one-hot encoding for DNA sequences

import numpy as np

BASE_TO_VECTOR = {
    "A": [1, 0, 0, 0],
    "C": [0, 1, 0, 0],
    "G": [0, 0, 1, 0],
    "T": [0, 0, 0, 1],
    "N": [0, 0, 0, 0]
}


def one_hot_encode(sequence):
    """
    Convert a DNA sequence into a one-hot encoded NumPy array.

    Output shape:
        (sequence_length, 4)
    """

    sequence = sequence.upper()

    encoded = np.array(
        [BASE_TO_VECTOR[base] for base in sequence],
        dtype=np.float32
    )

    return encoded

## Step 19 — Test One-Hot Encoding

A representative 500 bp window is encoded to verify that the one-hot
representation has the expected dimensions and nucleotide mappings.

The test checks:

- Output shape is `(500, 4)`
- Each standard nucleotide maps to exactly one active channel
- Ambiguous `N` bases are encoded as `[0, 0, 0, 0]`

In [24]:
# Test one-hot encoding on the first genome window

test_sequence = test_genome_windows[0]["Sequence"]

encoded_test = one_hot_encode(test_sequence)

print("Original sequence length:", len(test_sequence))
print("Encoded shape:", encoded_test.shape)

print("\nFirst 10 bases:")
print(test_sequence[:10])

print("\nFirst 10 encoded rows:")
print(encoded_test[:10])

print("\nUnique encoded vectors:")
print(np.unique(encoded_test, axis=0))

if encoded_test.shape == (500, 4):
    print("\n✓ One-hot encoding shape is correct.")
else:
    print("\n✗ Unexpected encoding shape.")

Original sequence length: 500
Encoded shape: (500, 4)

First 10 bases:
TCTTACTGCT

First 10 encoded rows:
[[0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]

Unique encoded vectors:
[[0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]

✓ One-hot encoding shape is correct.


## Step 20 — Validate Ambiguous Base Encoding

Ambiguous nucleotide bases are represented by `N`.

To avoid assigning an arbitrary nucleotide identity to an ambiguous base,
`N` is encoded as:

    N → [0, 0, 0, 0]

This test explicitly verifies that the encoder handles ambiguous bases
according to the defined representation.

In [25]:
# Test ambiguous N-base encoding

n_test_sequence = "ACGTN"

n_encoded = one_hot_encode(n_test_sequence)

print("Test sequence:", n_test_sequence)
print("\nEncoded representation:")
print(n_encoded)

print("\nEncoding of N:")
print(n_encoded[4])

if np.array_equal(n_encoded[4], [0, 0, 0, 0]):
    print("\n✓ N is correctly encoded as [0, 0, 0, 0].")
else:
    print("\n✗ N encoding is incorrect.")

Test sequence: ACGTN

Encoded representation:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]]

Encoding of N:
[0. 0. 0. 0.]

✓ N is correctly encoded as [0, 0, 0, 0].


## Step 21 — Validate Encoding Across the Test Genome

The one-hot encoder is tested across all windows generated from the
representative genome.

This verifies that:

- Every window produces the expected `(500, 4)` representation.
- All encoded values are valid binary values.
- No unexpected nucleotide encoding occurs.

The complete dataset will not be materialized in memory at this stage.
This test is restricted to one genome before scaling to the full cohort.

In [26]:
# Validate one-hot encoding across all windows of the test genome

encoded_windows = []

for window in test_genome_windows:
    encoded = one_hot_encode(window["Sequence"])
    encoded_windows.append(encoded)

# Convert to a single test array
encoded_windows_array = np.stack(encoded_windows)

print("Number of windows:", len(encoded_windows_array))
print("Encoded array shape:", encoded_windows_array.shape)

print("Minimum value:", encoded_windows_array.min())
print("Maximum value:", encoded_windows_array.max())

# Validate dimensions and values
shape_valid = encoded_windows_array.shape == (len(test_genome_windows), 500, 4)
values_valid = np.all(
    np.isin(encoded_windows_array, [0.0, 1.0])
)

if shape_valid and values_valid:
    print("\n✓ Genome-wide one-hot encoding validation passed.")
else:
    print("\n✗ Encoding validation failed.")

Number of windows: 10885
Encoded array shape: (10885, 500, 4)
Minimum value: 0.0
Maximum value: 1.0

✓ Genome-wide one-hot encoding validation passed.


## Step 22 — Define Streaming Window Batches

The complete genome cohort contains millions of overlapping windows.
Therefore, all windows will not be one-hot encoded and stored simultaneously.

Instead, windows from each genome are processed in manageable batches.

For each genome:

1. Generate its windows.
2. Divide the windows into batches.
3. One-hot encode each batch.
4. Process the batch through the CNN.
5. Retain the resulting window embeddings for genome-level aggregation.

This preserves the multiple-instance learning structure while controlling
memory usage during model training.

In [27]:
# Window batch size
WINDOW_BATCH_SIZE = 128


def generate_window_batches(windows, batch_size=WINDOW_BATCH_SIZE):
    """
    Yield one-hot encoded batches of genome windows.

    Each batch has shape:
        (batch_size, 500, 4)

    The final batch may contain fewer than batch_size windows.
    """
    
    for start in range(0, len(windows), batch_size):
        batch_windows = windows[start:start + batch_size]

        batch_encoded = np.stack([
            one_hot_encode(window["Sequence"])
            for window in batch_windows
        ])

        yield batch_encoded

In [28]:
print("Type of test_windows:", type(test_windows))
print("Value of test_windows:", test_windows if isinstance(test_windows, int) else "not an integer")

print("\nType of test_genome_windows:", type(test_genome_windows))
print("Type of test_windows:", type(test_windows))

Type of test_windows: <class 'int'>
Value of test_windows: 1911065

Type of test_genome_windows: <class 'list'>
Type of test_windows: <class 'int'>


In [29]:
# Use the actual window list from the representative genome
representative_windows = test_genome_windows

print("Type:", type(representative_windows))
print("Number of windows:", len(representative_windows))

Type: <class 'list'>
Number of windows: 10885


In [30]:
# Test streaming batches on the representative genome

batch_generator = generate_window_batches(
    representative_windows,
    batch_size=WINDOW_BATCH_SIZE
)

batch_count = 0
total_windows = 0

for batch in batch_generator:
    batch_count += 1
    total_windows += batch.shape[0]

    if batch_count <= 3:
        print(f"Batch {batch_count}:", batch.shape)

print("\nTotal batches:", batch_count)
print("Total windows processed:", total_windows)
print("Expected windows:", len(representative_windows))

if total_windows == len(representative_windows):
    print("\n✓ All windows were processed exactly once.")
else:
    print("\n✗ Window count mismatch.")

Batch 1: (128, 500, 4)
Batch 2: (128, 500, 4)
Batch 3: (128, 500, 4)

Total batches: 86
Total windows processed: 10885
Expected windows: 10885

✓ All windows were processed exactly once.


## Step 24 — Define the CNN Feature Extractor

The CNN operates on individual 500-bp genomic windows represented using
one-hot encoding.

Its purpose is to learn local sequence features such as nucleotide
patterns and motifs that may contribute to antimicrobial resistance.

The CNN does not directly produce the final genome-level prediction.

Instead, it transforms each genomic window into a lower-dimensional
feature representation (window embedding).

These window embeddings are subsequently aggregated using attention-based
multiple-instance learning to produce a genome-level prediction.

In [31]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 23 — CNN Feature Extraction

A convolutional neural network (CNN) is used to learn local sequence patterns
from one-hot encoded 500-bp genomic windows.

Each window is represented as:

    (500, 4)

where the four channels correspond to the nucleotide bases A, C, G, and T.

PyTorch Conv1D layers expect the input in the format:

    (batch_size, channels, sequence_length)

Therefore, each encoded window is rearranged from:

    (batch_size, 500, 4)

to:

    (batch_size, 4, 500)

The CNN processes each genomic window independently and produces a
128-dimensional embedding for every window.

These window-level embeddings are subsequently passed to the attention-based
multiple-instance learning (MIL) component, which aggregates information across
all windows belonging to the same genome.

The CNN is therefore a feature extractor rather than the final genome-level
classifier.

In [32]:
import torch
import torch.nn as nn


class GenomeCNN(nn.Module):
    """
    CNN feature extractor for one-hot encoded DNA windows.

    Input:
        (batch_size, 500, 4)

    Output:
        (batch_size, 128)

    Each 500-bp genomic window is converted into a
    128-dimensional learned embedding.
    """

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=4,
            out_channels=64,
            kernel_size=15,
            padding=7
        )

        self.conv2 = nn.Conv1d(
            in_channels=64,
            out_channels=128,
            kernel_size=15,
            padding=7
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        self.global_pool = nn.AdaptiveMaxPool1d(1)

        self.embedding = nn.Linear(
            128,
            128
        )

    def forward(self, x):
        """
        Forward pass.

        Input:
            x shape = (batch_size, 500, 4)

        Output:
            shape = (batch_size, 128)
        """

        # Rearrange for Conv1D:
        # (batch, 500, 4) → (batch, 4, 500)
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.pool(x)

        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # Global pooling across sequence positions
        x = self.global_pool(x)

        # Remove final sequence dimension
        x = x.squeeze(-1)

        # Produce final window embedding
        x = self.embedding(x)

        return x

In [33]:
# Create the CNN feature extractor

cnn = GenomeCNN()

print(cnn)

GenomeCNN(
  (conv1): Conv1d(4, 64, kernel_size=(15,), stride=(1,), padding=(7,))
  (conv2): Conv1d(64, 128, kernel_size=(15,), stride=(1,), padding=(7,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveMaxPool1d(output_size=1)
  (embedding): Linear(in_features=128, out_features=128, bias=True)
)


In [34]:
# Take the first 128 encoded windows from the representative genome

test_batch = encoded_windows_array[:128]

print("Input shape:", test_batch.shape)

# Convert NumPy array to PyTorch tensor
test_tensor = torch.tensor(
    test_batch,
    dtype=torch.float32
)

# Pass through CNN
with torch.no_grad():
    test_embeddings = cnn(test_tensor)

print("Output shape:", test_embeddings.shape)

Input shape: (128, 500, 4)
Output shape: torch.Size([128, 128])


## Step 25 — Attention-Based Multiple-Instance Learning

The CNN produces a learned embedding for each genomic window.

However, the final prediction is made at the genome level rather than the
window level.

Therefore, all windows belonging to a genome are treated as a single
multiple-instance learning (MIL) bag.

For a genome containing N windows:

    Window 1 → embedding 1
    Window 2 → embedding 2
    ...
    Window N → embedding N

An attention mechanism learns an importance weight for each window embedding.

The attention weights determine which genomic regions contribute more strongly
to the genome-level representation.

The weighted window embeddings are then aggregated into a single genome
embedding, which is used for Resistant/Susceptible classification.

This allows the model to learn which genomic regions are most informative for
ciprofloxacin resistance without assigning an independent resistance label
to every window.

In [35]:
import torch
import torch.nn as nn


class AttentionMIL(nn.Module):
    """
    Attention-based multiple-instance learning aggregator.

    Input:
        (number_of_windows, 128)

    Output:
        genome_embedding: (128,)
        attention_weights: (number_of_windows,)
        prediction_logit: scalar
    """

    def __init__(self, embedding_dim=128):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.classifier = nn.Linear(
            embedding_dim,
            1
        )

    def forward(self, embeddings):
        """
        embeddings:
            Tensor of shape (number_of_windows, embedding_dim)
        """

        # Calculate an attention score for every window
        attention_scores = self.attention(embeddings)

        # Convert scores into normalized attention weights
        attention_weights = torch.softmax(
            attention_scores,
            dim=0
        )

        # Weighted aggregation of all window embeddings
        genome_embedding = torch.sum(
            attention_weights * embeddings,
            dim=0
        )

        # Genome-level prediction
        prediction_logit = self.classifier(
            genome_embedding
        )

        return (
            genome_embedding,
            attention_weights.squeeze(-1),
            prediction_logit.squeeze(-1)
        )

In [36]:
# Create the attention-based MIL aggregator

mil = AttentionMIL(
    embedding_dim=128
)

print(mil)

AttentionMIL(
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (classifier): Linear(in_features=128, out_features=1, bias=True)
)


In [37]:
# Test attention MIL on the CNN embeddings

with torch.no_grad():
    genome_embedding, attention_weights, prediction_logit = mil(
        test_embeddings
    )

print("Window embeddings shape:", test_embeddings.shape)
print("Genome embedding shape:", genome_embedding.shape)
print("Attention weights shape:", attention_weights.shape)
print("Prediction logit shape:", prediction_logit.shape)

print("\nAttention weights sum:", attention_weights.sum().item())

Window embeddings shape: torch.Size([128, 128])
Genome embedding shape: torch.Size([128])
Attention weights shape: torch.Size([128])
Prediction logit shape: torch.Size([])

Attention weights sum: 1.0000001192092896


## Step 27 — Combined CNN and Attention-MIL Model

The complete model combines the CNN feature extractor with the
attention-based multiple-instance learning (MIL) aggregator.

For each genome:

1. Genomic windows are generated from the FASTA sequence.
2. Windows are one-hot encoded.
3. Windows are processed through the CNN in memory-efficient batches.
4. The CNN produces a learned embedding for every window.
5. Embeddings from all windows belonging to the genome are combined.
6. Attention MIL assigns an importance weight to each window.
7. The weighted embeddings are aggregated into a single genome-level
   representation.
8. A classifier predicts ciprofloxacin resistance or susceptibility.

The CNN therefore learns local sequence representations, while the
attention-MIL component learns which genomic regions are most informative
for the genome-level AMR phenotype.

In [38]:
class GenomeAMRModel(nn.Module):
    """
    Complete genome-level AMR prediction model.

    CNN:
        Converts each 500-bp DNA window into a 128-dimensional embedding.

    Attention MIL:
        Aggregates all window embeddings belonging to one genome and
        produces a genome-level prediction.
    """

    def __init__(self):
        super().__init__()

        self.cnn = GenomeCNN(
        )

        self.mil = AttentionMIL(
            embedding_dim=128
        )

    def forward(self, window_batch):
        """
        Input:
            window_batch:
                (number_of_windows, 500, 4)

        Output:
            genome_embedding:
                (128,)

            attention_weights:
                (number_of_windows,)

            prediction_logit:
                scalar
        """

        # Extract a learned representation for every window
        window_embeddings = self.cnn(
            window_batch
        )

        # Aggregate window representations at genome level
        genome_embedding, attention_weights, prediction_logit = self.mil(
            window_embeddings
        )

        return (
            genome_embedding,
            attention_weights,
            prediction_logit
        )

In [39]:
# Create the complete CNN + Attention-MIL model

model = GenomeAMRModel()

print(model)

GenomeAMRModel(
  (cnn): GenomeCNN(
    (conv1): Conv1d(4, 64, kernel_size=(15,), stride=(1,), padding=(7,))
    (conv2): Conv1d(64, 128, kernel_size=(15,), stride=(1,), padding=(7,))
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (global_pool): AdaptiveMaxPool1d(output_size=1)
    (embedding): Linear(in_features=128, out_features=128, bias=True)
  )
  (mil): AttentionMIL(
    (attention): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=1, bias=True)
    )
    (classifier): Linear(in_features=128, out_features=1, bias=True)
  )
)


## Step 28 — Test the Combined CNN + Attention-MIL Model

The complete model combines the CNN feature extractor and attention-based
multiple-instance learning into a single genome-level prediction pipeline.

A batch of one-hot encoded genomic windows is first processed by the shared
CNN to obtain window-level embeddings. These embeddings are then aggregated
by the attention-MIL module into a single genome-level representation and
prediction.

This test verifies that the complete forward pass works correctly before
training is implemented.

In [40]:
# Find the window-batching generator currently defined in the notebook

[name for name in globals() if "batch" in name.lower() and callable(globals()[name])]

['generate_window_batches']

In [41]:
# Test the complete CNN + Attention-MIL model

# Generate the first streaming batch from the representative genome
test_batch = next(
    generate_window_batches(
        representative_windows,
        batch_size=128
    )
)

print("Input batch shape:", test_batch.shape)

# Convert NumPy array to PyTorch tensor
test_tensor = torch.tensor(
    test_batch,
    dtype=torch.float32
)

# Run the complete CNN + Attention-MIL model
with torch.no_grad():
    genome_embedding, attention_weights, prediction_logit = model(test_tensor)

print("Genome embedding shape:", genome_embedding.shape)
print("Attention weights shape:", attention_weights.shape)
print("Prediction logit:", prediction_logit)
print("Prediction shape:", prediction_logit.shape)

Input batch shape: (128, 500, 4)
Genome embedding shape: torch.Size([128])
Attention weights shape: torch.Size([128])
Prediction logit: tensor(-0.0221)
Prediction shape: torch.Size([])


## Step 29 — Genome-Level Streaming Inference

The complete genome contains thousands of genomic windows, which cannot be processed
simultaneously without unnecessary memory usage.

The streaming pipeline processes genomic windows in batches:

1. Generate 500-bp windows from the genome.
2. One-hot encode a batch of windows.
3. Pass the batch through the CNN.
4. Collect the resulting 128-dimensional window embeddings.
5. Continue until all genome windows have been processed.
6. Aggregate the complete set of window embeddings using Attention-MIL.
7. Produce one genome-level prediction.

This allows the model to process complete bacterial genomes while keeping
memory usage bounded by the batch size.

In [42]:
# Process the complete representative genome in streaming batches

batch_size = 128

embedding_batches = []
total_windows_processed = 0

with torch.no_grad():

    for batch_number, batch in enumerate(
        generate_window_batches(
            representative_windows,
            batch_size=batch_size
        ),
        start=1
    ):

        # Convert NumPy batch to PyTorch tensor
        batch_tensor = torch.tensor(
            batch,
            dtype=torch.float32
        )

        # Extract a 128-dimensional embedding for each window
        batch_embeddings = model.cnn(batch_tensor)

        # Move embeddings to CPU and store them
        embedding_batches.append(
            batch_embeddings.cpu()
        )

        total_windows_processed += batch_tensor.shape[0]

        # Show the first few batches
        if batch_number <= 3:
            print(
                f"Batch {batch_number}: "
                f"input {batch_tensor.shape} → "
                f"embeddings {batch_embeddings.shape}"
            )

# Combine all window embeddings into one genome-level collection
all_window_embeddings = torch.cat(
    embedding_batches,
    dim=0
)

print("\nTotal windows processed:", total_windows_processed)
print("Expected windows:", len(representative_windows))
print(
    "All window embeddings shape:",
    all_window_embeddings.shape
)

if total_windows_processed == len(representative_windows):
    print("\n✓ Complete genome processed exactly once.")
else:
    print("\n✗ Window count mismatch.")

Batch 1: input torch.Size([128, 500, 4]) → embeddings torch.Size([128, 128])
Batch 2: input torch.Size([128, 500, 4]) → embeddings torch.Size([128, 128])
Batch 3: input torch.Size([128, 500, 4]) → embeddings torch.Size([128, 128])

Total windows processed: 10885
Expected windows: 10885
All window embeddings shape: torch.Size([10885, 128])

✓ Complete genome processed exactly once.


In [43]:
# Apply Attention-MIL to all window embeddings from the genome

with torch.no_grad():

    genome_embedding, attention_weights, prediction_logit = model.mil(
        all_window_embeddings
    )

print("Window embeddings shape:", all_window_embeddings.shape)
print("Genome embedding shape:", genome_embedding.shape)
print("Attention weights shape:", attention_weights.shape)
print("Prediction logit:", prediction_logit)
print("Attention weights sum:", attention_weights.sum().item())

Window embeddings shape: torch.Size([10885, 128])
Genome embedding shape: torch.Size([128])
Attention weights shape: torch.Size([10885])
Prediction logit: tensor(-0.0198)
Attention weights sum: 1.0000004768371582


In [44]:
# Validate the attention distribution

attention_np = attention_weights.detach().cpu().numpy()

print("Number of attention weights:", len(attention_np))
print("Minimum attention weight:", attention_np.min())
print("Maximum attention weight:", attention_np.max())
print("Mean attention weight:", attention_np.mean())
print("Sum of attention weights:", attention_np.sum())

# Check that all weights are non-negative
non_negative = np.all(attention_np >= 0)

# Check that they sum approximately to 1
sums_to_one = np.isclose(attention_np.sum(), 1.0)

print("\nNon-negative weights:", non_negative)
print("Weights sum to 1:", sums_to_one)

if non_negative and sums_to_one:
    print("\n✓ Attention distribution validation passed.")
else:
    print("\n✗ Attention distribution validation failed.")

Number of attention weights: 10885
Minimum attention weight: 8.836167e-05
Maximum attention weight: 9.421431e-05
Mean attention weight: 9.186959e-05
Sum of attention weights: 1.0000005

Non-negative weights: True
Weights sum to 1: True

✓ Attention distribution validation passed.


## Step 30 — Define Genome-Level Training Data Stream

During training, each genome is treated as one multiple-instance learning
(MIL) bag.

Windows are generated and one-hot encoded in batches rather than storing
the complete window dataset in memory.

For each genome:

1. Load the corresponding FASTA file.
2. Parse the genome into contigs.
3. Generate windows independently within each contig.
4. One-hot encode windows in batches.
5. Return the batches together with the genome-level label.

The genome label is applied only after all window embeddings have been
aggregated by Attention-MIL.

In [45]:
def generate_genome_training_batches(
    genome_id,
    batch_size=WINDOW_BATCH_SIZE
):
    """
    Stream one-hot encoded window batches from a single genome.

    The genome remains one MIL bag.
    Windows are generated contig-by-contig and processed in batches.

    Yields:
        batch_tensor:
            NumPy array of shape
            (batch_size, 500, 4)

        label:
            Genome-level binary phenotype label.
    """

    # Locate FASTA
    fasta_path = genome_dir / f"{genome_id}.fna"

    if not fasta_path.exists():
        raise FileNotFoundError(
            f"FASTA file not found for Genome ID: {genome_id}"
        )

    # Get genome-level label
    genome_row = model_df[
        model_df["Genome ID"] == genome_id
    ]

    if len(genome_row) != 1:
        raise ValueError(
            f"Expected exactly one phenotype record for {genome_id}, "
            f"found {len(genome_row)}"
        )

    label = int(genome_row.iloc[0]["Label"])

    # Parse genome
    contigs = parse_fasta(fasta_path)

    # Process contigs independently
    batch_windows = []

    for contig_id, sequence in contigs.items():

        # Generate windows from this contig
        contig_windows = generate_windows(
            sequence,
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        for window in contig_windows:

            batch_windows.append(
                window["Sequence"]
            )

            # Yield a full batch
            if len(batch_windows) == batch_size:

                batch_encoded = np.stack([
                    one_hot_encode(seq)
                    for seq in batch_windows
                ])

                yield batch_encoded, label

                batch_windows = []

    # Yield final partial batch
    if batch_windows:

        batch_encoded = np.stack([
            one_hot_encode(seq)
            for seq in batch_windows
        ])

        yield batch_encoded, label

In [46]:
# Test genome-level training batches

training_batch_count = 0
training_window_count = 0
training_labels = []

for batch, label in generate_genome_training_batches(
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
):

    training_batch_count += 1
    training_window_count += batch.shape[0]
    training_labels.append(label)

    if training_batch_count <= 3:
        print(
            f"Batch {training_batch_count}:",
            batch.shape,
            "| Label:",
            label
        )

print("\nTotal batches:", training_batch_count)
print("Total windows:", training_window_count)
print("Expected windows:", len(test_genome_windows))
print("Unique labels:", set(training_labels))

Batch 1: (128, 500, 4) | Label: 0
Batch 2: (128, 500, 4) | Label: 0
Batch 3: (128, 500, 4) | Label: 0

Total batches: 86
Total windows: 10885
Expected windows: 10885
Unique labels: {0}


## Step 31 — Define Genome-Level Training Forward Pass

A complete genome may contain thousands of windows, so windows are passed
through the CNN in smaller batches.

The resulting window embeddings are retained for the current genome and
then passed together to Attention-MIL.

The loss is therefore calculated from a single genome-level prediction,
preserving the MIL formulation.

In [47]:
def forward_genome(
    model,
    genome_id,
    batch_size=WINDOW_BATCH_SIZE,
    device="cpu"
):
    """
    Perform a differentiable forward pass for one genome.

    Returns:
        genome_embedding
        attention_weights
        prediction_logit
        label
    """

    embedding_batches = []
    label = None

    for batch, batch_label in generate_genome_training_batches(
        genome_id,
        batch_size=batch_size
    ):

        # Convert NumPy → PyTorch
        batch_tensor = torch.tensor(
            batch,
            dtype=torch.float32,
            device=device
        )

        # CNN forward pass
        batch_embeddings = model.cnn(
            batch_tensor
        )

        embedding_batches.append(
            batch_embeddings
        )

        label = batch_label

    # Combine all windows belonging to this genome
    all_embeddings = torch.cat(
        embedding_batches,
        dim=0
    )

    # Attention-MIL aggregation
    (
        genome_embedding,
        attention_weights,
        prediction_logit
    ) = model.mil(
        all_embeddings
    )

    return (
        genome_embedding,
        attention_weights,
        prediction_logit,
        label
    )

In [48]:
# Test differentiable genome-level forward pass

model.train()

(
    genome_embedding,
    attention_weights,
    prediction_logit,
    label
) = forward_genome(
    model,
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
)

print("Genome embedding:", genome_embedding.shape)
print("Attention weights:", attention_weights.shape)
print("Prediction logit:", prediction_logit.shape)
print("Genome label:", label)

print("\nNumber of windows:", len(attention_weights))
print("Attention sum:", attention_weights.sum().item())

Genome embedding: torch.Size([128])
Attention weights: torch.Size([10885])
Prediction logit: torch.Size([])
Genome label: 0

Number of windows: 10885
Attention sum: 1.0000004768371582


## Step 32 — Test Genome-Level Loss and Gradient Flow

The model predicts antibiotic resistance at the genome level.

Because the task is binary classification (Resistant vs Susceptible),
BCEWithLogitsLoss is used directly on the genome-level prediction logit.

This step verifies that gradients can flow through the complete model:

Genome windows → CNN → Attention MIL → classifier → loss.

In [49]:
# Step 32 — Test genome-level loss and gradient flow

model.train()

# Clear any existing gradients
model.zero_grad()

# Forward pass through one complete genome
(
    genome_embedding,
    attention_weights,
    prediction_logit,
    label
) = forward_genome(
    model,
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
)

# Convert genome label to PyTorch tensor
label_tensor = torch.tensor(
    float(label),
    dtype=torch.float32,
    device=prediction_logit.device
)

# Binary classification loss
criterion = torch.nn.BCEWithLogitsLoss()

loss = criterion(
    prediction_logit,
    label_tensor
)

print("Prediction logit:", prediction_logit.item())
print("Genome label:", label)
print("Loss:", loss.item())

# Backpropagation
loss.backward()

print("\nGradient flow verified.")

Prediction logit: -0.019812829792499542
Genome label: 0
Loss: 0.6832898259162903

Gradient flow verified.


In [50]:
# Check that trainable model parameters received gradients

gradient_parameters = 0
gradient_norm = 0.0

for name, parameter in model.named_parameters():

    if parameter.requires_grad and parameter.grad is not None:

        gradient_parameters += 1

        gradient_norm += (
            parameter.grad.detach().norm().item()
        ) ** 2

gradient_norm = gradient_norm ** 0.5

print("Parameters with gradients:", gradient_parameters)
print("Total gradient norm:", gradient_norm)

Parameters with gradients: 12
Total gradient norm: 1.814669316895361


## Step 33 — Test One-Genome Optimizer Step

The optimizer updates the CNN, Attention-MIL, and classifier parameters
using the genome-level loss.

A single training step is performed first to verify that model parameters
actually change after backpropagation and optimization.

In [51]:
# Step 33 — Define loss function and optimizer

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [52]:
# Test one complete optimization step

model.train()

# Clear old gradients
optimizer.zero_grad()

# Save one parameter before the update
parameter_name = "cnn.conv1.weight"

parameter_before = (
    model.cnn.conv1.weight.detach()
    .clone()
)

# Forward pass
(
    genome_embedding,
    attention_weights,
    prediction_logit,
    label
) = forward_genome(
    model,
    test_genome_id,
    batch_size=WINDOW_BATCH_SIZE
)

# Genome-level target
label_tensor = torch.tensor(
    float(label),
    dtype=torch.float32,
    device=prediction_logit.device
)

# Calculate loss
loss = criterion(
    prediction_logit,
    label_tensor
)

# Backpropagation
loss.backward()

# Update parameters
optimizer.step()

# Compare parameter after update
parameter_after = (
    model.cnn.conv1.weight.detach()
)

parameter_change = (
    parameter_after - parameter_before
).norm().item()

print("Genome label:", label)
print("Prediction logit:", prediction_logit.item())
print("Loss:", loss.item())
print("Parameter change:", parameter_change)

Genome label: 0
Prediction logit: -0.019812829792499542
Loss: 0.6832898259162903
Parameter change: 0.061958197504282


## Step 34 — Fresh Model Initialization and Training Setup

In [53]:
# Step 34 — Initialize the model for the actual experiment

import random
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import accuracy_score, roc_auc_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select available hardware
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Fresh model: discard the optimizer-test update
model = GenomeAMRModel().to(device)

# Training configuration
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Use the existing genome-level splits
train_genome_ids = train_df["Genome ID"].astype(str).tolist()
val_genome_ids = val_df["Genome ID"].astype(str).tolist()
test_genome_ids = test_df["Genome ID"].astype(str).tolist()

# Verify no leakage between partitions
assert set(train_genome_ids).isdisjoint(val_genome_ids)
assert set(train_genome_ids).isdisjoint(test_genome_ids)
assert set(val_genome_ids).isdisjoint(test_genome_ids)

assert set(train_df["Label"].unique()).issubset({0, 1})
assert set(val_df["Label"].unique()).issubset({0, 1})
assert set(test_df["Label"].unique()).issubset({0, 1})

print("Device:", device)
print("Training genomes:", len(train_genome_ids))
print("Validation genomes:", len(val_genome_ids))
print("Test genomes:", len(test_genome_ids))
print("Training labels:", train_df["Label"].value_counts().to_dict())
print("Validation labels:", val_df["Label"].value_counts().to_dict())
print("Fresh model initialized.")

Device: cpu
Training genomes: 796
Validation genomes: 171
Test genomes: 171
Training labels: {0: 464, 1: 332}
Validation labels: {0: 100, 1: 71}
Fresh model initialized.


## Step 35 — Genome-Level Training and Validation

In [54]:
# Step 35 — Genome-level training loop

EPOCHS = 10
BEST_MODEL_PATH = Path("models") / "best_amr_genome_cnn.pt"
BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):

    # -------------------------
    # Training
    # -------------------------
    model.train()
    shuffled_ids = train_genome_ids.copy()
    random.shuffle(shuffled_ids)

    train_losses = []

    for i, genome_id in enumerate(shuffled_ids, start=1):

        optimizer.zero_grad(set_to_none=True)

        _, _, logit, label = forward_genome(
            model,
            genome_id,
            batch_size=WINDOW_BATCH_SIZE,
            device=device
        )

        target = torch.tensor(
            float(label),
            dtype=torch.float32,
            device=device
        )

        loss = criterion(logit, target)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        if i % 25 == 0:
            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Train genomes: {i}/{len(shuffled_ids)}"
            )

    train_loss = float(np.mean(train_losses))

    # -------------------------
    # Validation
    # -------------------------
    model.eval()
    val_losses = []
    val_labels = []
    val_probs = []

    with torch.no_grad():

        for genome_id in val_genome_ids:

            _, _, logit, label = forward_genome(
                model,
                genome_id,
                batch_size=WINDOW_BATCH_SIZE,
                device=device
            )

            target = torch.tensor(
                float(label),
                dtype=torch.float32,
                device=device
            )

            val_losses.append(
                criterion(logit, target).item()
            )

            val_labels.append(int(label))
            val_probs.append(
                torch.sigmoid(logit).item()
            )

    val_loss = float(np.mean(val_losses))

    val_predictions = [
        int(prob >= 0.5) for prob in val_probs
    ]

    val_accuracy = accuracy_score(
        val_labels, val_predictions
    )

    val_auc = (
        roc_auc_score(val_labels, val_probs)
        if len(set(val_labels)) == 2
        else float("nan")
    )

    # Record epoch metrics
    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_auc": val_auc
    }

    history.append(epoch_result)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"Train loss:    {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.4f}")
    print(f"Validation ROC-AUC:  {val_auc:.4f}")

    # Save best checkpoint using validation loss
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_auc": val_auc,
                "seed": SEED
            },
            BEST_MODEL_PATH
        )

        print("Best model checkpoint saved.")

    print("-" * 50)

Epoch 1/10 | Train genomes: 25/796
Epoch 1/10 | Train genomes: 50/796
Epoch 1/10 | Train genomes: 75/796
Epoch 1/10 | Train genomes: 100/796
Epoch 1/10 | Train genomes: 125/796
Epoch 1/10 | Train genomes: 150/796
Epoch 1/10 | Train genomes: 175/796
Epoch 1/10 | Train genomes: 200/796


KeyboardInterrupt: 

In [55]:
import os
import psutil
import torch

memory = psutil.virtual_memory()

print("CPU cores available:", os.cpu_count())
print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CPU threads:", torch.get_num_threads())

CPU cores available: 12
Total RAM: 15.68 GB
Available RAM: 4.57 GB
PyTorch version: 2.14.0+cpu
CUDA available: False
PyTorch CPU threads: 10
